# Feature Engineering

Features are built in `src/features.py`, so every notebook uses the same code:

- `add_features` (Part A): promise per km, calendar (month sin/cos, weekday, hour), days to the next holiday, peak season, weekend days before the promised date, platform load.
- `add_seller_features` (Part B): seller history from `order_seller_bridge.csv.gz`. Point-in-time counts use every earlier order; outcome-based rates use only training orders already delivered at purchase time, smoothed towards the training average, and multi-seller orders take the worst seller.
- `FEATURES_LINEAR` / `FEATURES_TREE` and `build_preprocessor`: candidate feature sets and the preprocessing fitted inside each model pipeline.

This notebook builds both feature tables and checks them. Run `02_preprocessing.ipynb` first. Feature selection for Problem B is in `03c_feature_selection.ipynb`.

In [1]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
sys.path.append(str(ROOT))

from src.config import set_seed
from src.features import (
    FEATURES_LINEAR, FEATURES_TREE, LEAKAGE_COLUMNS, feature_columns, load_feature_table,
)

set_seed()

## 1. Build the feature tables

In [2]:
TARGETS = {"classification": "is_late", "regression": "delivery_days"}

tables = {problem: load_feature_table(problem) for problem in TARGETS}

for problem, table in tables.items():
    print(f"{problem:15s} {table.shape}")

classification  (96454, 72)
regression      (96454, 72)


## 2. Integrity checks

- One row per delivered order, the same orders and splits in both tables.
- Each table has only its own target.
- No column known only after purchase is a candidate feature, and every candidate exists.

In [3]:
classification, regression = tables["classification"], tables["regression"]

for table in tables.values():
    assert len(table) == 96_454
    assert table["order_id"].is_unique

pairs = classification[["order_id", "split"]].merge(
    regression[["order_id", "split"]], on="order_id", how="outer", validate="one_to_one", indicator=True
)
assert pairs["_merge"].eq("both").all()
assert pairs["split_x"].eq(pairs["split_y"]).all()

assert "delivery_days" not in classification and "is_late" not in regression

for name, features in {"FEATURES_LINEAR": FEATURES_LINEAR, "FEATURES_TREE": FEATURES_TREE}.items():
    columns = feature_columns(features)
    assert not set(columns) & set(LEAKAGE_COLUMNS), name
    for table in tables.values():
        missing = set(columns) - set(table.columns)
        assert not missing, (name, missing)
    print(f"{name}: {len(columns)} candidate columns, all present, none leaking")

FEATURES_LINEAR: 36 candidate columns, all present, none leaking
FEATURES_TREE: 63 candidate columns, all present, none leaking


## 3. Missing values

Distances and coordinates are missing for zip prefixes without a geolocation match (`geolocation_missing` flags them). `seller_past_avg_freight` is missing when no seller of the order has an earlier order, the payment columns for the one order without a payment record (`payment_missing`), `payment_per_installment` also for orders with 0 instalments, and `platform_load_ratio` for the very first order. All are imputed with the training median inside `build_preprocessor`.

In [4]:
candidates = feature_columns(FEATURES_TREE) + [
    c for c in feature_columns(FEATURES_LINEAR) if c not in feature_columns(FEATURES_TREE)
]
missing = regression[candidates].isna().sum()
display(missing[missing > 0].sort_values(ascending=False).to_frame("missing rows"))

,missing rows
seller_past_avg_freight,2881
promised_days_per_km,477
estimated_min_distance_km,477
estimated_mean_distance_km,477
estimated_max_distance_km,477
customer_zip_lat,265
customer_zip_lng,265
customer_zip_density,265
mean_seller_zip_lat,213
mean_seller_zip_lng,213


## 4. Engineered features by split

Means by split show which features drift over time. `platform_orders_7d` grows with the platform, which is why it is left out of `FEATURES_TREE`; `platform_load_ratio` is its stable replacement.

In [5]:
engineered = [
    "promised_days_per_km", "days_to_holiday", "peak_season_flag", "weekend_days_to_estimate",
    "platform_orders_7d", "platform_load_ratio",
    "seller_past_orders", "seller_backlog_7d", "seller_is_new", "seller_past_avg_freight",
    "seller_past_late_rate", "seller_past_delivery_days", "route_past_late_rate",
]
display(
    regression.groupby("split", sort=False)[engineered].mean().T.round(3)
)

split,train,validation,test
promised_days_per_km,0.201,0.158,0.131
days_to_holiday,25.359,51.519,26.040
peak_season_flag,0.055,0.000,0.000
weekend_days_to_estimate,7.092,5.630,4.415
platform_orders_7d,1345.597,1422.888,1892.386
platform_load_ratio,1.085,1.053,1.094
seller_past_orders,172.060,264.005,249.682
seller_backlog_7d,8.403,6.299,7.136
seller_is_new,0.108,0.091,0.110
seller_past_avg_freight,21.511,22.946,22.845
